In [1]:
from mltau.tools.evaluation import decode_ParTauDETR as dp
from mltau.tools.evaluation import set_to_set_models as sp
import awkward as ak

In [2]:
from hydra import compose, initialize
from omegaconf import OmegaConf

with initialize(version_base=None, config_path="../config", job_name="test_app"):
    cfg = compose(config_name="main_ParTauDETR")



In [3]:
from importlib import reload
reload(dp)
reload(sp)

<module 'mltau.tools.evaluation.set_to_set_models' from '/home/karl/ml-tau-model/mltau/tools/evaluation/set_to_set_models.py'>

In [4]:
import glob
import os.path

run_basename = '20260925_ParTauDETR_fixedLossWeights_weightSoftParentKinematics0p0_120epochs'
checkpoint_path = os.path.expanduser(f"~/runs/{run_basename}/models/ParTauDETR-model_best.ckpt")
dataset_dir = "/scratch/persistent/karl/ml-tau/0921_ParTauDETR_dataset"
signal_data_paths = sorted(glob.glob(f"{dataset_dir}/z_test_*.parquet"))
background_data_paths = sorted(glob.glob(f"{dataset_dir}/qq_test_*.parquet"))
assert signal_data_paths
assert background_data_paths

In [5]:
cfg.model.detr.loss.automatic_weight_optimization = False

In [6]:
import os
import shutil
import pyarrow.parquet as pq

obj_cls_trsh = 0.5
model = dp.load_model(checkpoint_path, cfg)


def merge_parquet_parts(part_paths, output_path):
    temporary_output_path = f'{output_path}.tmp'
    writer = None
    try:
        for part_path in part_paths:
            table = pq.read_table(part_path)
            if writer is None:
                writer = pq.ParquetWriter(temporary_output_path, table.schema)
            writer.write_table(table)
    finally:
        if writer is not None:
            writer.close()

    os.replace(temporary_output_path, output_path)


def run_inference_to_parquet(input_paths, sample_name, signal):
    parts_dir = os.path.expanduser(f'~/tmp/{run_basename}-{sample_name}-parts')
    output_dir = os.path.expanduser(f'~/tmp/{run_basename}-model_best-{obj_cls_trsh:.3f}'.replace('.', 'p') + 'thr')
    output_path = os.path.join(output_dir, f'{sample_name}_test_predictions.parquet')
    os.makedirs(parts_dir, exist_ok=True)
    os.makedirs(output_dir, exist_ok=True)
    part_paths = []

    try:
        for input_path in input_paths:
            outputs, targets, weights, reco_jet_p4s, data = dp.model_inference(
                checkpoint_path, input_path, cfg, model=model,
            )
            tau_tagging_score = dp.tau_scores(outputs).contiguous().numpy()

            if signal:
                true_daughters, pred_daughters = dp.create_predictions(
                    outputs, targets, weights, reco_jet_p4s, cfg,
                    obj_cls_trsh=obj_cls_trsh, tau_threshold=None,
                )
                data_to_save = sp.construct_prediction_file_content(
                    data, pred_daughters, true_daughters, cfg.dataset.tau_daughter_pdg_ids,
                    tau_tagging_score=tau_tagging_score, debug=True,
                )
            else:
                data_to_save = sp.construct_background_prediction_file_content(
                    data, tau_tagging_score=tau_tagging_score,
                )

            part_path = os.path.join(parts_dir, os.path.basename(input_path))
            ak.to_parquet(data_to_save, part_path)
            part_paths.append(part_path)
            del outputs, targets, weights, reco_jet_p4s, data, tau_tagging_score, data_to_save

        merge_parquet_parts(part_paths, output_path)
    finally:
        shutil.rmtree(parts_dir, ignore_errors=True)

    return output_path

In [7]:
signal_output_path = run_inference_to_parquet(
    signal_data_paths, sample_name='z', signal=True,
 )

background_output_path = run_inference_to_parquet(
    background_data_paths, sample_name='qq', signal=False,
 )

Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 90,793 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
R

In [8]:
print('Signal predictions:', signal_output_path)
print('Background predictions:', background_output_path)

Signal predictions: /home/karl/tmp/20260925_ParTauDETR_fixedLossWeights_weightSoftParentKinematics0p0_120epochs-model_best-0p500thr/z_test_predictions.parquet
Background predictions: /home/karl/tmp/20260925_ParTauDETR_fixedLossWeights_weightSoftParentKinematics0p0_120epochs-model_best-0p500thr/qq_test_predictions.parquet
